# The Scikit-Learn Pattern: Quy Trình 5 Bước Xây Dựng Model Machine Learning
Mô hình phân tích dữ liệu phim Netflix từ file `netflix_titles_cleaned.csv` theo chuẩn 5 bước Scikit-Learn:
1. **Load**: `pd.read_csv()` - Đọc dữ liệu vào DataFrame
2. **Split**: `train_test_split()` - Tách dữ liệu Train / Test
3. **Fit**: `model.fit(X, y)` - Huấn luyện mô hình tìm mối liên hệ giữa nội dung và độ tuổi khán giả
4. **Predict**: `model.predict(X)` - Áp dụng mô hình để dự đoán trên tập kiểm thử và dữ liệu mới
5. **Score**: `model.score(X, y)` - Đánh giá độ chính xác của mô hình

## Bước 1: Load (`pd.read_csv`)
- Đọc file `netflix_titles_cleaned.csv` vào pandas DataFrame.
- Chuẩn hóa cột `rating` thành 3 phân khúc khán giả: `Adults (18+)`, `Teens (13-14+)`, `Kids & Family` làm biến mục tiêu $y$.
- Kết hợp các trường văn bản `type`, `listed_in`, `description` thành đặc trưng đầu vào $X$.

In [ ]:
import pandas as pd
import numpy as np

# 1. Đọc dữ liệu
df = pd.read_csv('netflix_titles_cleaned.csv')
print(f"Kích thước dữ liệu gốc: {df.shape[0]} dòng, {df.shape[1]} cột")

# 2. Xây dựng biến mục tiêu y (Phân khúc khán giả)
def map_audience_segment(rating):
    if rating in ['TV-MA', 'R', 'NC-17']:
        return 'Adults (18+)'
    elif rating in ['TV-14', 'PG-13']:
        return 'Teens (13-14+)'
    elif rating in ['TV-PG', 'PG', 'TV-Y7', 'TV-Y', 'TV-G', 'G', 'TV-Y7-FV']:
        return 'Kids & Family'
    return 'Other'

df['target_segment'] = df['rating'].apply(map_audience_segment)
df_model = df[df['target_segment'] != 'Other'].copy()

# 3. Tạo đặc trưng X (Features)
df_model['features'] = df_model['type'] + ' | ' + df_model['listed_in'] + ' | ' + df_model['description']
X = df_model['features']
y = df_model['target_segment']

print(f"Số lượng mẫu hợp lệ: {len(df_model)} tác phẩm")
print("\nPhân bố các phân khúc khán giả:")
print(y.value_counts(normalize=True).mul(100).round(2).astype(str) + '%')
df_model[['title', 'rating', 'target_segment']].head()

## Bước 2: Split (`train_test_split`)
- Tách tập dữ liệu thành 2 phần độc lập:
  - **Train set (80%)**: Dùng để huấn luyện mô hình.
  - **Test set (20%)**: Giữ lại độc lập để đánh giá khách quan, tránh rò rỉ dữ liệu (data leakage).
- Thiết lập `stratify=y` để tỷ lệ phân bố nhãn ở hai tập cân bằng.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Tập huấn luyện (X_train): {len(X_train)} mẫu (80%)")
print(f"Tập kiểm tra   (X_test) : {len(X_test)} mẫu (20%)")

## Bước 3: Fit (`model.fit`)
- Sử dụng `Pipeline` kết hợp:
  1. `TfidfVectorizer`: Chuyển đổi văn bản thành ma trận trọng số từ vựng TF-IDF.
  2. `LogisticRegression`: Bộ phân loại đa lớp tuyến tính.
- Gọi `model.fit(X_train, y_train)` để mô hình học quan hệ giữa nội dung và phân khúc khán giả.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Thiết lập Pipeline
model = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=8000, ngram_range=(1, 2))),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, C=2.0))
])

# Huấn luyện mô hình
model.fit(X_train, y_train)
print("Huấn luyện thành công: model.fit(X_train, y_train)")

# Trích xuất các xu hướng từ khóa nổi bật theo từng phân khúc độ tuổi
tfidf_step = model.named_steps['tfidf']
clf_step = model.named_steps['classifier']
feature_names = tfidf_step.get_feature_names_out()

print("\n--- XU HƯỚNG TỪ KHÓA NỘI DUNG (Film Trends) ---")
for idx, cls in enumerate(clf_step.classes_):
    top_indices = clf_step.coef_[idx].argsort()[-6:][::-1]
    top_words = [feature_names[i] for i in top_indices]
    print(f"Nhóm [{cls}]: {', '.join(top_words)}")

## Bước 4: Predict (`model.predict`)
- Áp dụng mô hình đã huấn luyện để phân loại nhãn trên tập kiểm thử: `model.predict(X_test)`.
- Thử nghiệm mô hình trên các tác phẩm mới chưa từng có trong tập dữ liệu (Inference test).

In [ ]:
# 1. Dự đoán trên tập kiểm tra
y_pred = model.predict(X_test)
print(f"Đã dự đoán xong {len(y_pred)} mẫu trong tập Test.\n")

# 2. Thử nghiệm trên các bộ phim thực tế mới
new_movies = [
    {
        'title': 'Magic Forest Pals',
        'text': 'Movie | Children & Family Movies | A friendly pony and animated animals go on a cheerful journey to save the forest.'
    },
    {
        'title': 'Cartel Outlaws',
        'text': 'TV Show | Crime TV Shows, Action | A ruthless cartel boss executes violent drug murders and gang assassinations.'
    },
    {
        'title': 'Prom Night Crush',
        'text': 'Movie | Comedies, Romantic Movies | High school teenagers navigate first love, awkward friendships, and graduation dance.'
    }
]

print("--- THỬ NGHIỆM DỰ ĐOÁN PHIM MỚI ---")
for movie in new_movies:
    pred = model.predict([movie['text']])[0]
    prob = np.max(model.predict_proba([movie['text']])) * 100
    print(f"- Phim: '{movie['title']}'")
    print(f"  => Dự đoán: {pred} (Độ tin cậy: {prob:.1f}%)")

## Bước 5: Score (`model.score`)
- Sử dụng hàm `model.score(X_test, y_test)` để tính độ chính xác trung bình (Mean Accuracy).
- Hiển thị chi tiết Precision, Recall, F1-Score và ma trận nhầm lẫn Confusion Matrix.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Điểm chính xác theo phương thức .score()
accuracy = model.score(X_test, y_test)
print(f"Mean Accuracy (model.score): {accuracy:.4f} ({accuracy * 100:.2f}%)\n")

# 2. Báo cáo phân loại chi tiết
print("--- Classification Report ---")
print(classification_report(y_test, y_pred, digits=4))

# 3. Ma trận nhầm lẫn (Confusion Matrix)
print("--- Confusion Matrix ---")
labels = clf_step.classes_
cm = pd.DataFrame(confusion_matrix(y_test, y_pred, labels=labels), index=labels, columns=labels)
display(cm)

---\n## Bước Kiểm Thử (Testing): Kiểm Tra Tính Đúng Đắn Của Model\nThực thi bộ test kiểm thử tự động với các tiêu chí:\n1. **Tính toàn vẹn cấu trúc**: Không NaN/Null, kích thước khớp 100%, nhãn hợp lệ.\n2. **Tính chất xác suất**: Tổng xác suất các lớp = 1.0, nhãn dự đoán khớp argmax xác suất.\n3. **Độ nhạy ngữ cảnh (Semantic accuracy)**: Phim hoạt hình -> Kids, Phim bạo lực -> Adults, Phim học đường -> Teens.\n4. **Ngưỡng hiệu năng (Benchmark)**: Accuracy >= 60%, Recall Adults >= 70%, Precision Kids >= 75%.\n5. **Độ bền (Robustness & Edge cases)**: Không crash khi gặp dữ liệu rỗng, ký tự đặc biệt.

In [ ]:
from test_model import run_custom_test_suite\n\n# Chạy toàn bộ 15 bài kiểm thử tự động\nrun_custom_test_suite()